# 🔬 Exploration & Inspection of Friend's Scraped Papers (`data/papers.parquet`)

This notebook provides an in-depth inspection of the parquet file (`data/papers.parquet`) shared by your friend/collaborator.
It uses **pandas** to inspect the data types, metadata, document parsing metrics, nested section structures, and cleaned full-text layout, while comparing it to our project's Medallion Silver Lakehouse schema.

In [2]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Configure pandas display options for rich terminal & notebook output
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 1000)

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.5


## 1. Load the Parquet File

We locate and load `data/papers.parquet`, handling path resolution whether running from the repository root or the `notebooks/` directory.

In [4]:
# Robust path resolution
possible_paths = [
    Path("data/papers.parquet"),
    Path("../data/papers.parquet"),
]

parquet_path = None
for p in possible_paths:
    if p.exists():
        parquet_path = p
        break

if not parquet_path:
    raise FileNotFoundError("Could not locate data/papers.parquet! Please ensure it is placed in the data/ folder.")

print(f"Loading parquet dataset from: {parquet_path.resolve()}")
df = pd.read_parquet(parquet_path)
print(f"Dataset Shape: {df.shape[0]} row(s), {df.shape[1]} columns\n")
df.head()

Loading parquet dataset from: C:\document\Study documents\Uth-Data-Mining\data\papers.parquet
Dataset Shape: 1 row(s), 16 columns



paper_id                                                                       title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            abstract                                                     authors categories primary_category                   published_date                        crawled_at                               pdf_url                           html_url  total_sections  total_math_count  total_words                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## 2. Dataset Schema & Column Specifications

Here we inspect each column's name, inferred data type, null counts, and preview sample values.

In [6]:
schema_summary = pd.DataFrame({
    "Column": df.columns,
    "Dtype": [str(t) for t in df.dtypes.values],
    "Non-Null Count": df.notnull().sum().values,
    "Null Count": df.isnull().sum().values,
    "Sample Value": [
        str(df[col].iloc[0])[:65] + "..." if len(str(df[col].iloc[0])) > 65 else str(df[col].iloc[0])
        for col in df.columns
    ]
})
schema_summary

,Column,Dtype,Non-Null Count,Null Count,Sample Value
0,paper_id,str,1,0,2609.38282
1,title,str,1,0,Improving OCR Faithfulness via Gated and Attenuated On-Policy Dis...
2,abstract,str,1,0,Vision-language models may rewrite anomalous text in images into ...
3,authors,object,1,0,"['Baode Wang, Zuming Huang, Kexuan Ren, Jun Huang, Wei Chu']"
4,categories,object,1,0,['cs.AI']
5,primary_category,str,1,0,cs.AI
6,published_date,str,1,0,"Thu, 01 Oct 2026 00:00:00 -0400"
7,crawled_at,str,1,0,2026-10-02T02:08:18.154017+00:00
8,pdf_url,str,1,0,https://arxiv.org/pdf/2609.38282.pdf
9,html_url,str,1,0,https://arxiv.org/html/2609.38282


## 3. Paper Metadata Inspection

Extract bibliographic information: Paper ID, Title, Authors, Categories, Published Date, and URLs.

In [8]:
sample_row = df.iloc[0]

print("=" * 85)
print(f"📄 Paper ID         : {sample_row.get('paper_id')}")
print(f"📌 Title            : {sample_row.get('title')}")
print(f"👥 Authors          : {', '.join(sample_row.get('authors', []))}")
print(f"🏷️  Categories       : {sample_row.get('categories')}")
print(f"⭐ Primary Category : {sample_row.get('primary_category')}")
print(f"📅 Published Date   : {sample_row.get('published_date')}")
print(f"🌐 PDF URL          : {sample_row.get('pdf_url')}")
print(f"🔗 HTML URL         : {sample_row.get('html_url')}")
print("=" * 85)
print("\n📝 Abstract:\n")
print(sample_row.get("abstract"))

📄 Paper ID         : 2609.38282
📌 Title            : Improving OCR Faithfulness via Gated and Attenuated On-Policy Distillation
👥 Authors          : Baode Wang, Zuming Huang, Kexuan Ren, Jun Huang, Wei Chu
🏷️  Categories       : ['cs.AI']
⭐ Primary Category : cs.AI
📅 Published Date   : Thu, 01 Oct 2026 00:00:00 -0400
🌐 PDF URL          : https://arxiv.org/pdf/2609.38282.pdf
🔗 HTML URL         : https://arxiv.org/html/2609.38282

📝 Abstract:

Vision-language models may rewrite anomalous text in images into linguistically plausible expressions, compromising OCR transcription faithfulness. Sequence-level task rewards and local teacher guidance are complementary, but guidance from the same teacher may not remain equally effective as the student improves. Offline analysis shows that supervision from a fixed teacher becomes progressively less favorable as the student improves, both across training checkpoints and across response groups with different task rewards. Motivated by this observati

## 4. Quantitative Parsing & Content Metrics

Inspect the quantitative extraction statistics produced during crawling: number of sections, word count, math equations, and crawl timestamps.

In [10]:
metrics_df = pd.DataFrame([
    {"Metric": "Total Sections", "Value": sample_row.get("total_sections")},
    {"Metric": "Total Words", "Value": f"{sample_row.get('total_words'):,} words"},
    {"Metric": "Total Math Count", "Value": f"{sample_row.get('total_math_count'):,} LaTeX/math expressions"},
    {"Metric": "Crawled At", "Value": sample_row.get("crawled_at")},
    {"Metric": "Transformed At", "Value": sample_row.get("transformed_at")}
])
metrics_df

,Metric,Value
0,Total Sections,6
1,Total Words,"3,382 words"
2,Total Math Count,219 LaTeX/math expressions
3,Crawled At,2026-10-02T02:08:18.154017+00:00
4,Transformed At,2026-10-02T02:22:50.682100+00:00


## 5. Decomposing the Nested Section Hierarchy (`sections_json`)

The column `sections_json` contains structured JSON with individual sections, section titles, and text blocks.

In [12]:
raw_sections = sample_row.get("sections_json")
sections = json.loads(raw_sections) if isinstance(raw_sections, str) else raw_sections

print(f"Decomposed {len(sections)} structured sections from 'sections_json':\n")

section_table = []
for s in sections:
    sec_title = s.get("section_title", "Untitled")
    sec_id = s.get("section_id", "")
    text_content = s.get("section_text", "")
    char_len = len(text_content)
    word_count = len(text_content.split())
    section_table.append({
        "Section ID": sec_id,
        "Section Title": sec_title,
        "Word Count": word_count,
        "Character Count": char_len,
        "Preview": text_content[:85].replace("\n", " ") + "..." if char_len > 85 else text_content
    })

sec_df = pd.DataFrame(section_table)
sec_df

Decomposed 6 structured sections from 'sections_json':



,Section ID,Section Title,Word Count,Character Count,Preview
0,S1,1 Introduction,0,0,
1,S2,2 Related Work,0,0,
2,S3,3 Methods,0,0,
3,S4,4 Experiments,0,0,
4,S5,5 Analysis,0,0,
5,S6,6 Conclusions,0,0,


## 6. Clean Full Text Preview (`clean_full_text`)

Inspect the continuous full text parsed from the paper.

In [14]:
full_text = sample_row.get("clean_full_text", "")
print(f"Total full text length: {len(full_text):,} characters ({len(full_text.split()):,} words)\n")
print("-" * 40 + " FIRST 750 CHARACTERS OF CLEAN FULL TEXT " + "-" * 40)
print(full_text[:750])
print("\n" + "-" * 35 + " [TRUNCATED FOR DISPLAY] " + "-" * 35)

Total full text length: 25,791 characters (3,412 words)

---------------------------------------- FIRST 750 CHARACTERS OF CLEAN FULL TEXT ----------------------------------------
Title: Improving OCR Faithfulness via Gated and Attenuated On-Policy Distillation

Abstract: Vision-language models may rewrite anomalous text in images into linguistically plausible expressions, compromising OCR transcription faithfulness. Sequence-level task rewards and local teacher guidance are complementary, but guidance from the same teacher may not remain equally effective as the student improves. Offline analysis shows that supervision from a fixed teacher becomes progressively less favorable as the student improves, both across training checkpoints and across response groups with different task rewards. Motivated by this observation, we introduce GAD-RL, which adaptively regulates teacher supervision during joint post-training acco

----------------------------------- [TRUNCATED FOR DISPLAY] ---------

## 7. Schema Comparison with Our Project's Silver Lakehouse

Comparison between your friend's arXiv-based schema and our project's OpenAlex-based Silver Lakehouse (`data/silver/papers.parquet`).

In [16]:
silver_path = Path("data/silver/papers.parquet")
if not silver_path.exists():
    silver_path = Path("../data/silver/papers.parquet")

if silver_path.exists():
    silver_df = pd.read_parquet(silver_path)
    print(f"Our Project Silver Papers: {len(silver_df):,} records, {len(silver_df.columns)} columns\n")
    
    comp = pd.DataFrame({
        "Feature / Aspect": [
            "Primary Data Source",
            "Corpus Volume",
            "Full Text Included",
            "Parsed Sections Structure",
            "Citation Counts & In-Degree",
            "Citation Network Edges",
            "Keywords / Concepts",
            "Intended Phase Role"
        ],
        "Friend's File (data/papers.parquet)": [
            "arXiv HTML / PDF crawl",
            f"{len(df)} paper(s)",
            "Yes ('clean_full_text')",
            "Yes ('sections_json')",
            "Not tracked",
            "None",
            "arXiv categories ('cs.AI')",
            "Full-text parsing & chunking reference (Phase 2)"
        ],
        "Our Silver Lakehouse (data/silver/papers.parquet)": [
            "OpenAlex Polite API",
            f"{len(silver_df):,} papers (2017-2026)",
            "Bronze PDF Vault (194 vaulted, 9.8k metadata)",
            "To be produced in Phase 2",
            "Yes ('citation_count', PageRank)",
            "Yes (86,295 internal edges)",
            "Yes (63,659 keyword associations)",
            "Longitudinal graph trend mining (Phase 3) & RAG"
        ]
    })
    display(comp)
else:
    print("data/silver/papers.parquet not found for comparison.")

Our Project Silver Papers: 10,000 records, 17 columns



,Feature / Aspect,Friend's File (data/papers.parquet),Our Silver Lakehouse (data/silver/papers.parquet)
0,Primary Data Source,arXiv HTML / PDF crawl,OpenAlex Polite API
1,Corpus Volume,1 paper(s),"10,000 papers (2017-2026)"
2,Full Text Included,Yes ('clean_full_text'),"Bronze PDF Vault (194 vaulted, 9.8k metadata)"
3,Parsed Sections Structure,Yes ('sections_json'),To be produced in Phase 2
4,Citation Counts & In-Degree,Not tracked,"Yes ('citation_count', PageRank)"
5,Citation Network Edges,None,"Yes (86,295 internal edges)"
6,Keywords / Concepts,arXiv categories ('cs.AI'),"Yes (63,659 keyword associations)"
7,Intended Phase Role,Full-text parsing & chunking reference (Phase 2),Longitudinal graph trend mining (Phase 3) & RAG
